In [1]:
from astropy.io import fits
from pathlib import Path
import numpy as np
import sys
import numpy
from matplotlib.pyplot import figure, show, subplots
import numpy.ma as ma


datadir = "/net/virgo01/data/users/oassenberg/ObsAs/i"  #my data folder
 
#this sets the path that the program has to follow to get to our data!
path = Path(datadir)
filepaths_ = []
for filepath in path.iterdir():
    filepaths_.append(filepath)

filepaths = np.array(filepaths_)

filepaths.sort()
print(filepaths[0])
print(filepaths[-2])

#285 - 295 -> Biases 

#363 and 637 and 609 and 758 missing!
BIAS = []

for i in range(0, 10):
    BIAS.append(filepaths[i])
for i in range(798-289, 809-289):
    BIAS.append(filepaths[i])


DARK = []

for i in range(11, 17):
    DARK.append(filepaths[i])
for i in range(783-289, 796-289):
    DARK.append(filepaths[i])


FLATB = []

for i in range(303-285, 307-285):
    FLATB.append(filepaths[i])
for i in range(812-289, 814-289):
    FLATB.append(filepaths[i])


FLATV = []

for i in range(308-285, 312-285):
    FLATV.append(filepaths[i])
for i in range(815-289, 817-289):
    FLATV.append(filepaths[i])


FLATR = []

for i in range(314-285, 319-285):
    FLATR.append(filepaths[i])
for i in range(809-289, 811-289):
    FLATR.append(filepaths[i])


LIGHTR = []
LIGHTV = []
LIGHTB = []


for i in range(363-285, 604-286, 3):
    LIGHTR.append(filepaths[i])
for i in range(365-286, 604-286, 3):
    LIGHTV.append(filepaths[i])
for i in range(366-286, 604-286, 3):
    LIGHTB.append(filepaths[i])

for i in range(616-287, 636-287, 3):
    LIGHTR.append(filepaths[i])
for i in range(617-287, 636-287, 3):
    LIGHTV.append(filepaths[i])
for i in range(618-287, 636-287, 3):
    LIGHTB.append(filepaths[i])
    
for i in range(638-288, 758-289, 3):
    LIGHTR.append(filepaths[i])
for i in range(639-288, 758-289, 3):
    LIGHTV.append(filepaths[i])
for i in range(640-288, 758-289, 3):
    LIGHTB.append(filepaths[i])



GD336 = []
for i in range(759-289, 774-289):
    GD336.append(filepaths[i])

#Oli group is BV
#Grace group is VR

/net/virgo01/data/users/oassenberg/ObsAs/i/260430_LI_.00000285.Mouse_click_position.BIAS.FIT
/net/virgo01/data/users/oassenberg/ObsAs/i/260430_LI_.00000817.17h03m03.8s_69d55m49sN.FLAT.FIT


In [ ]:
BIAS_file = []


for i in range(len(BIAS)):
    hdul = fits.open(BIAS[i])
    BIAS_file.append(hdul[0].data)

datacube = np.stack(BIAS_file)
print(f"Shape of data cube with stacked images: {datacube.shape}")


stacked_BIAS = np.median(datacube, axis=0)
fig, ax = subplots()


j = ax.imshow(stacked_BIAS, vmax = np.percentile(stacked_BIAS, 99), vmin = np.percentile(stacked_BIAS, 1), origin = 'lower')
ax.set_axis_off()
ax.set_title('Master Bias Frame')
fig.colorbar(j, orientation = 'horizontal')
ax.invert_yaxis()




In [ ]:
DARKR = []
DARKB = []
DARKV = []
for i in range(-15, -10):
    hdul = fits.open(DARK[i])
    DARKR.append((hdul[0].data - stacked_BIAS)/15)

for i in range(-10, -5):
    hdul = fits.open(DARK[i])
    DARKB.append((hdul[0].data - stacked_BIAS)/25)

for i in range(-5, 0):
    hdul = fits.open(DARK[i])
    DARKV.append((hdul[0].data - stacked_BIAS)/20)


datacubeR = np.stack(DARKR)
datacubeB = np.stack(DARKB)
datacubeV = np.stack(DARKV)


Stack_DARKR = np.median(datacubeR, axis=0)
Stack_DARKB = np.median(datacubeB, axis=0)
Stack_DARKV = np.median(datacubeV, axis=0)

fig1, ax1 = subplots(1,2, figsize = (15,4))
ax1 = ax1.flatten()


ax1[0].set_title('Dark frame in B band')
DarkB = ax1[0].imshow(Stack_DARKB, vmax = np.percentile(Stack_DARKB, 95), vmin = np.percentile(Stack_DARKB, 5), origin = 'lower')
fig1.colorbar(DarkB, orientation = 'horizontal')
ax1[0].set_axis_off()

ax1[1].set_title('Dark frame in V band')
DarkV = ax1[1].imshow(Stack_DARKV, vmax = np.percentile(Stack_DARKV, 95), vmin = np.percentile(Stack_DARKV, 5), origin = 'lower')
fig1.colorbar(DarkV, orientation = 'horizontal')
ax1[1].set_axis_off()

In [ ]:
FLATB_2 = []
FLATV_2 = []


for i in range(3, len(FLATB)):
    hdul = fits.open(FLATB[i])
    Corr = (hdul[0].data - stacked_BIAS - Stack_DARKB)
    FLATB_2.append((Corr)/np.median(Corr))

FLATB_corr = np.median(np.stack(FLATB_2), axis = 0)
                       
for i in range(3, len(FLATV)):
    hdul = fits.open(FLATV[i])
    Corr = (hdul[0].data - stacked_BIAS - Stack_DARKV)
    FLATV_2.append((Corr)/np.median(Corr))

FLATV_corr = np.median(np.stack(FLATV_2), axis = 0)


In [ ]:
fig2, ax2 = subplots(1,2, figsize = (15,4))
ax2 = ax2.flatten()


imB = ax2[0].imshow(FLATB_corr, vmax = np.percentile(FLATB_corr, 99), vmin = np.percentile(FLATB_corr, 1), origin = 'lower')
fig2.colorbar(imB, orientation = 'horizontal')
ax2[0].set_axis_off()
ax2[0].set_title('FLat field in B filter')


imV = ax2[1].imshow(FLATV_corr, vmax = np.percentile(FLATV_corr, 99), vmin = np.percentile(FLATV_corr, 1), origin = 'lower')
fig2.colorbar(imV, orientation = 'horizontal')
ax2[1].set_axis_off()
ax2[1].set_title('FLat field in V filter')

In [ ]:
# Reducing data

LIGHTB_reduced = []
LIGHTV_reduced = []

for lightb in LIGHTB:
    hdul = fits.open(lightb)
    res = (hdul[0].data - stacked_BIAS - Stack_DARKB*25)/FLATB_corr
    LIGHTB_reduced.append(res)

for lightv in LIGHTV:
    hdul = fits.open(lightv)
    res = (hdul[0].data - stacked_BIAS - Stack_DARKV*20)/FLATV_corr
    LIGHTV_reduced.append(res)

In [ ]:
#use astroalign register
#data reduce before aligning 